In [3]:
import pandas as pd
import re
import contractions
import nltk
import spacy

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [2]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 9.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.1 MB/s eta 0:00:00


In [4]:
!pip install nltk spacy

In [5]:
# ============================================================
# 1. NLTK SETUP
# ============================================================

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [6]:

# ============================================================
# 2. LOAD SPACY MODEL
# ============================================================

nlp = spacy.load("en_core_web_sm")

In [7]:
# ============================================================
# 3. LOAD DATASET
# ============================================================

df = pd.read_csv("/kaggle/input/datasets/mobeenfatimah/fake-news-detection-dataset-6000-news-articles/news_dataset.csv")

print("Dataset shape:")
print(df.shape)

print("\nColumns:")
print(df.columns)

print("\nFirst 5 rows:")
print(df.head())


Dataset shape:
(6000, 5)

Columns:
Index(['title', 'text', 'subject', 'date', 'label'], dtype='object')

First 5 rows:
                                               title  \
0   WATCH: Trump Just Told All The Anti-Gay Bigot...   
1  China backs U.N. call for justice in Yemen, U....   
2  THE PEOPLE’S PRESIDENT: Trump Meets With Coal ...   
3  WSJ REPORTER RIPS INTO DEM CANDIDATES For Thei...   
4  Lawmakers aim to delay U.S. ceding control of ...   

                                                text       subject  \
0  A whole lot of evangelical Trump voters just d...          News   
1  GENEVA (Reuters) - China signaled on Wednesday...     worldnews   
2                                                         politics   
3  THE WSJ S MARY KISSEL NAILS IT ON THE DEM DEBA...      politics   
4  WASHINGTON (Reuters) - Critics of a plan for t...  politicsNews   

                  date  label  
0    November 13, 2016      0  
1  September 13, 2017       1  
2         Feb 16, 2017     

In [9]:
# ============================================================
# 4. STOP WORDS
# ============================================================

stop_words = set(stopwords.words("english"))

# Keep "not" because it can be important for meaning
stop_words.discard("not")


In [10]:
# ============================================================
# 5. TEXT CLEANING FUNCTION
# ============================================================

def clean_text(text):

    # Convert missing values / other values to string
    text = str(text)

    # ----------------------------------------
    # Expand contractions
    # ----------------------------------------
    # don't -> do not
    # can't -> cannot
    # he's -> he is
    # they're -> they are
    text = contractions.fix(text)

    # ----------------------------------------
    # Convert to lowercase
    # ----------------------------------------
    text = text.lower()

    # ----------------------------------------
    # Remove URLs
    # ----------------------------------------
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    # ----------------------------------------
    # Remove HTML tags
    # ----------------------------------------
    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    # ----------------------------------------
    # Remove email addresses
    # ----------------------------------------
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # ----------------------------------------
    # Remove numbers
    # ----------------------------------------
    text = re.sub(
        r"\d+",
        " ",
        text
    )

    # ----------------------------------------
    # Remove punctuation / special characters
    # ----------------------------------------
    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # ----------------------------------------
    # Remove extra spaces
    # ----------------------------------------
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()



In [11]:
# ============================================================
# 6. TOKENIZATION + STOP WORD REMOVAL
# ============================================================

def tokenize_and_remove_stopwords(text):

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stop words
    tokens = [
        word
        for word in tokens
        if word not in stop_words
    ]

    return tokens



In [12]:
# ============================================================
# 7. LEMMATIZATION
# ============================================================

def lemmatize_tokens(tokens):

    # Convert tokens back into text
    text = " ".join(tokens)

    # Process with spaCy
    doc = nlp(text)

    # Get lemma
    lemmas = [
        token.lemma_
        for token in doc
        if not token.is_punct
        and not token.is_space
    ]

    return lemmas



In [13]:
# ============================================================
# 8. COMPLETE PREPROCESSING FUNCTION
# ============================================================

def preprocess_text(text):

    # Step 1: Cleaning
    text = clean_text(text)

    # Step 2: Tokenization
    tokens = tokenize_and_remove_stopwords(text)

    # Step 3: Lemmatization
    lemmas = lemmatize_tokens(tokens)

    # Step 4: Convert list back to string
    return " ".join(lemmas)



In [14]:
# ============================================================
# 9. HANDLE MISSING VALUES
# ============================================================

df["text"] = df["text"].fillna("")


# ============================================================
# 10. APPLY PREPROCESSING
# ============================================================

print("\nProcessing text...")

df["clean_text"] = df["text"].apply(preprocess_text)



Processing text...


In [15]:
# ============================================================
# 11. DISPLAY ORIGINAL VS CLEANED TEXT
# ============================================================

print("\nOriginal vs Cleaned Text:\n")

for i in range(min(5, len(df))):

    print("--------------------------------------------------")

    print("ORIGINAL:")
    print(df["text"].iloc[i])

    print("\nCLEANED:")
    print(df["clean_text"].iloc[i])




Original vs Cleaned Text:

--------------------------------------------------
ORIGINAL:
A whole lot of evangelical Trump voters just discovered they got duped.By picking Mike Pence as his vice-president, Donald Trump sent a message to the LGBT community that their rights are in jeopardy of being rolled back.In fact, one of the reasons why so many conservative  Christians  tossed their alleged  morality  out the window to vote for him was precisely because they believed he would pass a constitutional amendment banning same-sex marriage or pack the courts with anti-gay bigots to strike down all the rulings in favor of marriage equality.But during his interview on 60 minutes with Leslie Stahl on Sunday night, Trump once again backed off another campaign promise. Do you support marriage equality?  Stahl asked Trump. It s irrelevant because it s already settled,  Trump replied.  It s law. It was settled in the Supreme Court. I mean, it s done. When Stahl pointed out that Trump s Supreme Co

In [17]:
# ============================================================
# 13. FINAL DATASET INFORMATION
# ============================================================

print("\nFinal dataset shape:")
print(df.shape)

print("\nFinal columns:")
print(df.columns)

print("\nSample processed data:")
print(
    df[
        ["text", "clean_text", "label"]
    ].head()
)


Final dataset shape:
(6000, 6)

Final columns:
Index(['title', 'text', 'subject', 'date', 'label', 'clean_text'], dtype='object')

Sample processed data:
                                                text  \
0  A whole lot of evangelical Trump voters just d...   
1  GENEVA (Reuters) - China signaled on Wednesday...   
2                                                      
3  THE WSJ S MARY KISSEL NAILS IT ON THE DEM DEBA...   
4  WASHINGTON (Reuters) - Critics of a plan for t...   

                                          clean_text  label  
0  whole lot evangelical trump voter discover get...      0  
1  geneva reuters china signal wednesday willing ...      1  
2                                                         0  
3  wsj mary kissel nail dem debate three candidat...      0  
4  washington reuters critic plan government cede...      1  


In [18]:
# ============================================================
# 12. SAVE PROCESSED DATASET
# ============================================================

df.to_csv(
    "news_dataset_processed.csv",
    index=False
)

print("\nProcessing completed.")

print("\nProcessed dataset saved as:")
print("news_dataset_processed.csv")



Processing completed.

Processed dataset saved as:
news_dataset_processed.csv
